# 02 — Construction du Gold Standard ABSA V2

## Objectif

Construire depuis les **400 000 avis bruts FR/EN** un Gold Standard bilingue,
traçable et reproductible pour les expériences ABSA V2.

Chaque avis est représenté par quatre aspects indépendants :

- `quality`
- `price`
- `delivery`
- `service`

Chaque aspect prend l'un des états :

`absent / négatif / neutre / positif`

Les étoiles peuvent servir au sampling, mais **ne constituent pas la vérité terrain ABSA**.

La stratégie suivie est progressive :

1. pilote naturel ;
2. analyse de la couverture ;
3. enrichissement ciblé des aspects minoritaires ;
4. annotation sémantique ;
5. construction du Gold final ;
6. split multilabel DEV / TEST.


In [1]:
from pathlib import Path

import re

import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# Le notebook se trouve dans ABSA-V2/notebooks/
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
GOLD_DIR = DATA_DIR / "gold"
INTERIM_DIR = DATA_DIR / "interim"

RAW_FILES = {
    "fr_full": RAW_DIR / "df_fr_full.jsonl",
    "en_full": RAW_DIR / "df_en_full.jsonl",
}

for name, path in RAW_FILES.items():
    relative_path = path.relative_to(PROJECT_ROOT)
    print(f"{name:15} | existe={path.exists()} | {relative_path}")

datasets = {
    "fr_full": pd.read_json(RAW_FILES["fr_full"], lines=True),
    "en_full": pd.read_json(RAW_FILES["en_full"], lines=True),
}

print("\nFR :", datasets["fr_full"].shape)
print("EN :", datasets["en_full"].shape)


fr_full         | existe=True | data\raw\df_fr_full.jsonl
en_full         | existe=True | data\raw\df_en_full.jsonl

FR : (200000, 4)
EN : (200000, 4)


## 1. Construction du pilote naturel — 100 avis

Le pilote est équilibré par langue et par nombre d'étoiles :

- 50 avis FR ;
- 50 avis EN ;
- 10 avis par étoile et par langue.

Son objectif n'est pas d'équilibrer les aspects, mais d'observer leur fréquence dans
un échantillon sélectionné sans filtre ABSA.


In [2]:
SEED = 42
N_PER_STAR_PER_LANGUAGE = 10

def build_pilot_sample(df, language):
    """
    Construit un échantillon équilibré par note (label 0-4)
    pour une langue donnée.
    """
    sampled_parts = []

    for label in sorted(df["label"].unique()):
        subset = df[df["label"] == label].copy()

        sampled = subset.sample(
            n=N_PER_STAR_PER_LANGUAGE,
            random_state=SEED
        )

        sampled["language"] = language
        sampled_parts.append(sampled)

    return pd.concat(sampled_parts, ignore_index=True)


pilot_fr = build_pilot_sample(
    datasets["fr_full"],
    language="fr"
)

pilot_en = build_pilot_sample(
    datasets["en_full"],
    language="en"
)

pilot = pd.concat(
    [pilot_fr, pilot_en],
    ignore_index=True
)

# Mélange final reproductible
pilot = pilot.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

# Passage du label 0-4 vers étoiles 1-5
pilot["stars"] = pilot["label"] + 1

# Colonnes utiles uniquement
pilot = pilot[
    [
        "id",
        "language",
        "text",
        "stars",
        "label_text"
    ]
].copy()

print("Shape :", pilot.shape)

print("\nRépartition par langue :")
print(pilot["language"].value_counts())

print("\nRépartition langue × étoiles :")
print(
    pd.crosstab(
        pilot["language"],
        pilot["stars"]
    )
)

print("\nIDs dupliqués :", pilot["id"].duplicated().sum())
print("Textes dupliqués :", pilot["text"].duplicated().sum())

display(pilot.head())

Shape : (100, 5)

Répartition par langue :
language
en    50
fr    50
Name: count, dtype: int64

Répartition langue × étoiles :
stars      1   2   3   4   5
language                    
en        10  10  10  10  10
fr        10  10  10  10  10

IDs dupliqués : 0
Textes dupliqués : 0


,id,language,text,stars,label_text
0,en_0166548,en,I purchased these to put on frozen foods and t...,4,3
1,en_0561880,en,Received questionable email from Amazon.com pe...,1,0
2,en_0963371,en,So so\n\nNot very satisfied with this product,3,2
3,fr_0858022,fr,"Bien étudié\n\nImpressionnant, essaie dans une...",5,4
4,fr_0524797,fr,"Tasse ideale,ne coule pas\n\nCette tasse est t...",5,4


In [ ]:
PILOT_PATH = INTERIM_DIR / "pilot_absa_v2_100.csv"

pilot.to_csv(
   PILOT_PATH,
   index=False,
    encoding="utf-8-sig"
)

print("Pilote sauvegardé :", PILOT_PATH.relative_to(PROJECT_ROOT))

Pilote sauvegardé : data\interim\pilot_absa_v2_100.csv


## 2. Analyse du pilote annoté

Le pilote annoté sert à mesurer :

- la distribution des états par aspect ;
- la fréquence de présence de chaque aspect ;
- le nombre d'aspects par avis ;
- la fréquence des cas multi-aspects.

Cette analyse détermine si un enrichissement ciblé est nécessaire.


# Expérience 1 — Analyse du pilote annoté V2

## Objectif

Mesurer la distribution naturelle des aspects et sentiments sur le pilote avant de décider de la stratégie de construction du Gold V2.

Ces résultats ne sont pas utilisés pour entraîner un modèle.

In [5]:
ANNOTATED_PILOT_PATH = INTERIM_DIR / "pilot_absa_v2_100_annotated_final.csv"
pilot_v2 = pd.read_csv(ANNOTATED_PILOT_PATH)

print("Shape :", pilot_v2.shape)
print("Colonnes :", pilot_v2.columns.tolist())

Shape : (100, 9)
Colonnes : ['id', 'language', 'text', 'stars', 'label_text', 'quality', 'price', 'delivery', 'service']


In [6]:
ASPECTS = ["quality", "price", "delivery", "service"]

print("=== DISTRIBUTION DES LABELS PAR ASPECT ===")

for aspect in ASPECTS:
    counts = pilot_v2[aspect].value_counts()
    percentages = (
        pilot_v2[aspect]
        .value_counts(normalize=True)
        .mul(100)
        .round(1)
    )

    result = pd.DataFrame({
        "nombre": counts,
        "pourcentage": percentages
    })

    print(f"\n--- {aspect.upper()} ---")
    print(result)

=== DISTRIBUTION DES LABELS PAR ASPECT ===

--- QUALITY ---
         nombre  pourcentage
quality                     
négatif      46         46.0
positif      40         40.0
absent       12         12.0
neutre        2          2.0

--- PRICE ---
         nombre  pourcentage
price                       
absent       88         88.0
négatif       8          8.0
positif       3          3.0
neutre        1          1.0

--- DELIVERY ---
          nombre  pourcentage
delivery                     
absent        88         88.0
négatif        8          8.0
positif        4          4.0

--- SERVICE ---
         nombre  pourcentage
service                     
absent       90         90.0
négatif       6          6.0
positif       4          4.0


In [7]:
print("=== PRÉSENCE DES ASPECTS ===")

for aspect in ASPECTS:
    present = (pilot_v2[aspect] != "absent").sum()

    print(
        f"{aspect:10} : "
        f"{present:3} / {len(pilot_v2)} "
        f"({present / len(pilot_v2) * 100:.1f} %)"
    )

=== PRÉSENCE DES ASPECTS ===
quality    :  88 / 100 (88.0 %)
price      :  12 / 100 (12.0 %)
delivery   :  12 / 100 (12.0 %)
service    :  10 / 100 (10.0 %)


In [8]:
pilot_v2["nb_aspects"] = (
    pilot_v2[ASPECTS] != "absent"
).sum(axis=1)

print("=== NOMBRE D'ASPECTS PAR AVIS ===")
print(pilot_v2["nb_aspects"].value_counts().sort_index())

print(
    "\nAvis sans aspect :",
    (pilot_v2["nb_aspects"] == 0).sum()
)

print(
    "Avis mono-aspect :",
    (pilot_v2["nb_aspects"] == 1).sum()
)

print(
    "Avis multi-aspects :",
    (pilot_v2["nb_aspects"] >= 2).sum()
)

=== NOMBRE D'ASPECTS PAR AVIS ===
nb_aspects
0     1
1    76
2    23
Name: count, dtype: int64

Avis sans aspect : 1
Avis mono-aspect : 76
Avis multi-aspects : 23


### Décision

Le sampling naturel est conservé pour représenter la population, mais il ne fournit pas
suffisamment d'exemples pour certains aspects minoritaires.

On ajoute donc une **présélection ciblée** pour Prix, Livraison et Service.

Les règles lexicales ci-dessous servent uniquement à trouver des **candidats à annoter**.
Elles ne déterminent jamais les labels ABSA finaux.


## 3. Construction et validation des pools candidats


In [9]:
# ============================================================
# POOLS DE CANDIDATS POUR LES ASPECTS MINORITAIRES
# ============================================================

KEYWORDS = {
    "price": {
        "fr": [
            r"\bprix\b",
            r"\bcher\b",
            r"\bchère\b",
            r"\bcoût\b",
            r"\bcoûte\b",
            r"\beuros?\b",
            r"\b€\b",
            r"\brapport qualité.?prix\b",
        ],
        "en": [
            r"\bprice\b",
            r"\bexpensive\b",
            r"\bcheap\b",
            r"\bcost\b",
            r"\bworth\b",
            r"\bvalue for money\b",
            r"\bvalue\b",
            r"\bdollars?\b",
        ],
    },

    "delivery": {
        "fr": [
            r"\blivraison\b",
            r"\blivré\b",
            r"\blivrée\b",
            r"\blivreur\b",
            r"\bcolis\b",
            r"\breçu\b",
            r"\breçue\b",
            r"\barrivé\b",
            r"\barrivée\b",
            r"\btransport\b",
            r"\bretard\b",
        ],
        "en": [
            r"\bdelivery\b",
            r"\bdelivered\b",
            r"\bshipping\b",
            r"\bshipment\b",
            r"\bpackage\b",
            r"\barrived\b",
            r"\breceived\b",
            r"\blate\b",
            r"\bcourier\b",
        ],
    },

    "service": {
        "fr": [
            r"\bservice client\b",
            r"\bsav\b",
            r"\bvendeur\b",
            r"\bremboursement\b",
            r"\bremboursé\b",
            r"\bremboursée\b",
            r"\bretour\b",
            r"\bcontacté\b",
            r"\bcontact\b",
            r"\bsupport\b",
        ],
        "en": [
            r"\bcustomer service\b",
            r"\bseller\b",
            r"\brefund\b",
            r"\brefunded\b",
            r"\breturn\b",
            r"\breplacement\b",
            r"\bsupport\b",
            r"\bcontacted\b",
        ],
    },
}

In [10]:
def keyword_candidate(text, patterns):
    text = str(text).lower()

    return any(
        re.search(pattern, text, flags=re.IGNORECASE)
        for pattern in patterns
    )

In [11]:
fr_pool = datasets["fr_full"][["id", "text", "label"]].copy()
fr_pool["language"] = "fr"

en_pool = datasets["en_full"][["id", "text", "label"]].copy()
en_pool["language"] = "en"

full_pool = pd.concat(
    [fr_pool, en_pool],
    ignore_index=True
)

full_pool["stars"] = full_pool["label"] + 1

print("Pool total :", len(full_pool))

Pool total : 400000


In [12]:
for aspect in ["price", "delivery", "service"]:

    full_pool[f"candidate_{aspect}"] = False

    for language in ["fr", "en"]:

        mask_language = full_pool["language"] == language

        full_pool.loc[
            mask_language,
            f"candidate_{aspect}"
        ] = (
            full_pool.loc[mask_language, "text"]
            .apply(
                lambda text: keyword_candidate(
                    text,
                    KEYWORDS[aspect][language]
                )
            )
        )


print("=== CANDIDATS TROUVÉS ===")

for aspect in ["price", "delivery", "service"]:

    mask = full_pool[f"candidate_{aspect}"]

    print(
        f"{aspect:10} : "
        f"{mask.sum():,} "
        f"({mask.mean() * 100:.2f} %)"
    )

=== CANDIDATS TROUVÉS ===
price      : 53,451 (13.36 %)
delivery   : 58,305 (14.58 %)
service    : 29,025 (7.26 %)


In [13]:
SEED = 42
N_PER_LANGUAGE = 25

candidate_samples = []

for aspect in ["price", "delivery", "service"]:
    for language in ["fr", "en"]:

        subset = full_pool[
            (full_pool["language"] == language)
            & (full_pool[f"candidate_{aspect}"])
        ].copy()

        sample = subset.sample(
            n=N_PER_LANGUAGE,
            random_state=SEED
        )

        sample["target_candidate"] = aspect

        candidate_samples.append(sample)

candidate_validation = pd.concat(
    candidate_samples,
    ignore_index=True
)

print("Nombre de sélections :", len(candidate_validation))
print("\nRépartition :")
print(
    pd.crosstab(
        candidate_validation["target_candidate"],
        candidate_validation["language"]
    )
)

print(
    "\nIDs uniques :",
    candidate_validation["id"].nunique()
)

print(
    "Sélections dupliquées entre pools :",
    len(candidate_validation)
    - candidate_validation["id"].nunique()
)

Nombre de sélections : 150

Répartition :
language          en  fr
target_candidate        
delivery          25  25
price             25  25
service           25  25

IDs uniques : 150
Sélections dupliquées entre pools : 0


In [ ]:
CANDIDATE_VALIDATION_PATH = (
    INTERIM_DIR / "candidate_filter_validation_150.csv"
)

candidate_validation[
    [
        "id",
        "language",
        "text",
        "stars",
        "target_candidate"
    ]
].to_csv(
    CANDIDATE_VALIDATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Sauvegardé :", CANDIDATE_VALIDATION_PATH.relative_to(PROJECT_ROOT))

Sauvegardé : data\interim\candidate_filter_validation_150.csv


## 4. Gold V2 — Batch 1 de 400 avis

Le premier batch combine :

- **200 avis naturels** : 100 FR + 100 EN ;
- **200 avis enrichis** : Prix, Livraison et Service.

Les IDs du pilote et de la validation des filtres sont exclus afin d'éviter le leakage.


200 naturels
├── 100 FR
└── 100 EN

200 enrichis
├── 70 Prix
├── 70 Livraison
└── 60 Service

In [16]:
# ============================================================
# 1. IDs déjà utilisés à exclure
# ============================================================

USED_IDS = set()

# Pilote 100
USED_IDS.update(
    pilot_v2["id"].astype(str)
)

# Validation filtres 150
USED_IDS.update(
    candidate_validation["id"].astype(str)
)

print("IDs déjà utilisés :", len(USED_IDS))

IDs déjà utilisés : 250


In [17]:
# ============================================================
# 2. ÉCHANTILLON NATUREL — 200 avis
# ============================================================

natural_pool = full_pool[
    ~full_pool["id"].astype(str).isin(USED_IDS)
].copy()

natural_samples = []

for language in ["fr", "en"]:

    subset = natural_pool[
        natural_pool["language"] == language
    ].copy()

    # 20 avis par étoile × 5 étoiles = 100 par langue
    for star in [1, 2, 3, 4, 5]:

        star_subset = subset[
            subset["stars"] == star
        ]

        sample = star_subset.sample(
            n=20,
            random_state=SEED
        )

        sample["sampling_source"] = "natural"
        sample["target_candidate"] = "none"

        natural_samples.append(sample)

natural_gold = pd.concat(
    natural_samples,
    ignore_index=True
)

print("Natural :", len(natural_gold))

print(
    pd.crosstab(
        natural_gold["language"],
        natural_gold["stars"]
    )
)

Natural : 200
stars      1   2   3   4   5
language                    
en        20  20  20  20  20
fr        20  20  20  20  20


In [18]:
# ============================================================
# 3. ÉCHANTILLON ENRICHI — 200 avis
# ============================================================

# On exclut aussi les 200 naturels
used_after_natural = (
    USED_IDS
    | set(natural_gold["id"].astype(str))
)

targets = {
    "price": {
        "fr": 35,
        "en": 35,
    },
    "delivery": {
        "fr": 35,
        "en": 35,
    },
    "service": {
        "fr": 30,
        "en": 30,
    },
}

enriched_samples = []

CURRENT_USED = set(used_after_natural)

for aspect, languages in targets.items():

    for language, n_samples in languages.items():

        subset = full_pool[
            (full_pool["language"] == language)
            & (full_pool[f"candidate_{aspect}"])
            & (~full_pool["id"].astype(str).isin(CURRENT_USED))
        ].copy()

        sample = subset.sample(
            n=n_samples,
            random_state=SEED
        )

        sample["sampling_source"] = "enriched"
        sample["target_candidate"] = aspect

        enriched_samples.append(sample)

        # évite qu'un même avis soit sélectionné ensuite
        # pour un autre pool
        CURRENT_USED.update(
            sample["id"].astype(str)
        )

enriched_gold = pd.concat(
    enriched_samples,
    ignore_index=True
)

print("Enriched :", len(enriched_gold))

print(
    pd.crosstab(
        enriched_gold["target_candidate"],
        enriched_gold["language"]
    )
)

Enriched : 200
language          en  fr
target_candidate        
delivery          35  35
price             35  35
service           30  30


In [19]:
# ============================================================
# 4. GOLD V2 — PREMIÈRE TRANCHE 400
# ============================================================

gold_v2_batch1 = pd.concat(
    [
        natural_gold,
        enriched_gold
    ],
    ignore_index=True
)

gold_v2_batch1 = gold_v2_batch1.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

gold_v2_batch1 = gold_v2_batch1[
    [
        "id",
        "language",
        "text",
        "stars",
        "sampling_source",
        "target_candidate"
    ]
].copy()

print("Shape :", gold_v2_batch1.shape)

print(
    "\nRépartition sampling_source :"
)
print(
    gold_v2_batch1["sampling_source"]
    .value_counts()
)

print(
    "\nRépartition langue :"
)
print(
    gold_v2_batch1["language"]
    .value_counts()
)

print(
    "\nIDs dupliqués :",
    gold_v2_batch1["id"]
    .duplicated()
    .sum()
)

print(
    "Textes dupliqués :",
    gold_v2_batch1["text"]
    .duplicated()
    .sum()
)
assert len(gold_v2_batch1) == 400
assert gold_v2_batch1["id"].duplicated().sum() == 0
assert gold_v2_batch1["text"].duplicated().sum() == 0


Shape : (400, 6)

Répartition sampling_source :
sampling_source
enriched    200
natural     200
Name: count, dtype: int64

Répartition langue :
language
fr    200
en    200
Name: count, dtype: int64

IDs dupliqués : 0
Textes dupliqués : 0


In [20]:
# ============================================================
# 5. CONTRÔLE DE LEAKAGE
# ============================================================

new_ids = set(
    gold_v2_batch1["id"].astype(str)
)

overlap = new_ids & USED_IDS

print(
    "Chevauchement avec les 250 déjà étudiés :",
    len(overlap)
)
assert len(overlap) == 0


Chevauchement avec les 250 déjà étudiés : 0


In [ ]:
GOLD_V2_BATCH1_PATH = (
    GOLD_DIR / "gold_v2_batch1_400_unannotated.csv"
)

gold_v2_batch1.to_csv(
    GOLD_V2_BATCH1_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Gold batch 1 sauvegardé :",
    GOLD_V2_BATCH1_PATH
)

Gold batch 1 sauvegardé : c:\Users\youne\OneDrive\Desktop\ABSA-V2\data\gold\gold_v2_batch1_400_unannotated.csv


## 5. Enrichissement ciblé — Batch 2 de 300 avis

Après analyse du premier batch, la couverture de Qualité est déjà suffisante.
Le second enrichissement cible donc principalement :

- Prix ;
- Livraison ;
- Service.

Le filtre Service est renforcé afin de réduire les faux positifs liés aux termes ambigus
comme `return`, `support` ou `replacement`.

Le Batch 2 contient :

- 100 candidats Prix ;
- 60 candidats Livraison ;
- 120 candidats Service V2 ;
- 20 avis naturels.

Soit **300 nouveaux avis**.


In [ ]:
# ============================================================
# GOLD V2 — BATCH 2 : 300 nouveaux avis
# 100 Prix + 60 Livraison + 120 Service V2 + 20 Naturels
# ============================================================


SEED = 42

# ------------------------------------------------------------
# 1. Filtre Service V2
# ------------------------------------------------------------

SERVICE_FILTER_V2 = {
    "fr": {
        "strong": [
            r"\bservice client\b",
            r"\bsav\b",
            r"\bvendeur\b",
            r"\bcontacté\b",
        ],
        "ambiguous": [
            r"\bremboursement\b",
            r"\bremboursé\b",
            r"\bremboursée\b",
            r"\bretour\b",
            r"\bsupport\b",
        ],
        "context": [
            r"\bamazon\b",
            r"\bvendeur\b",
            r"\bservice\b",
            r"\bclient\b",
            r"\bcontact\b",
            r"\bréponse\b",
            r"\brepond",
            r"\brépond",
            r"\baide\b",
        ],
    },

    "en": {
        "strong": [
            r"\bcustomer service\b",
            r"\bseller\b",
            r"\bcontacted\b",
        ],
        "ambiguous": [
            r"\brefund\b",
            r"\brefunded\b",
            r"\breturn\b",
            r"\breplacement\b",
            r"\bsupport\b",
        ],
        "context": [
            r"\bamazon\b",
            r"\bseller\b",
            r"\bcustomer\b",
            r"\bservice\b",
            r"\bcontact\b",
            r"\bresponse\b",
            r"\breplied\b",
            r"\breply\b",
            r"\bhelp\b",
        ],
    },
}


def service_candidate_v2(text, language):

    text = str(text)
    rules = SERVICE_FILTER_V2[language]

    strong_match = any(
        re.search(pattern, text, flags=re.IGNORECASE)
        for pattern in rules["strong"]
    )

    if strong_match:
        return True

    ambiguous_match = any(
        re.search(pattern, text, flags=re.IGNORECASE)
        for pattern in rules["ambiguous"]
    )

    context_match = any(
        re.search(pattern, text, flags=re.IGNORECASE)
        for pattern in rules["context"]
    )

    return ambiguous_match and context_match


full_pool["candidate_service_v2"] = full_pool.apply(
    lambda row: service_candidate_v2(
        row["text"],
        row["language"]
    ),
    axis=1
)

# ------------------------------------------------------------
# 2. Exclure tout ce qu'on a déjà étudié
# ------------------------------------------------------------

USED_ALL = set()

# pilote 100
USED_ALL.update(
    pilot_v2["id"].astype(str)
)

# validation 150
USED_ALL.update(
    candidate_validation["id"].astype(str)
)

# batch 1 des 400
USED_ALL.update(
    gold_v2_batch1["id"].astype(str)
)

print("IDs déjà utilisés :", len(USED_ALL))

# ------------------------------------------------------------
# 3. Targets du Batch 2
# ------------------------------------------------------------

TARGETS = {
    "price": {
        "fr": 50,
        "en": 50,
    },

    "delivery": {
        "fr": 30,
        "en": 30,
    },

    "service_v2": {
        "fr": 60,
        "en": 60,
    },
}

selected_parts = []

CURRENT_USED = set(USED_ALL)

# ------------------------------------------------------------
# 4. Prix
# ------------------------------------------------------------

for language, n in TARGETS["price"].items():

    subset = full_pool[
        (full_pool["language"] == language)
        & full_pool["candidate_price"]
        & ~full_pool["id"].astype(str).isin(CURRENT_USED)
    ].copy()

    sample = subset.sample(
        n=n,
        random_state=SEED
    )

    sample["sampling_source"] = "enriched"
    sample["target_candidate"] = "price"

    selected_parts.append(sample)

    CURRENT_USED.update(
        sample["id"].astype(str)
    )

# ------------------------------------------------------------
# 5. Livraison
# ------------------------------------------------------------

for language, n in TARGETS["delivery"].items():

    subset = full_pool[
        (full_pool["language"] == language)
        & full_pool["candidate_delivery"]
        & ~full_pool["id"].astype(str).isin(CURRENT_USED)
    ].copy()

    sample = subset.sample(
        n=n,
        random_state=SEED
    )

    sample["sampling_source"] = "enriched"
    sample["target_candidate"] = "delivery"

    selected_parts.append(sample)

    CURRENT_USED.update(
        sample["id"].astype(str)
    )

# ------------------------------------------------------------
# 6. Service V2
# ------------------------------------------------------------

for language, n in TARGETS["service_v2"].items():

    subset = full_pool[
        (full_pool["language"] == language)
        & full_pool["candidate_service_v2"]
        & ~full_pool["id"].astype(str).isin(CURRENT_USED)
    ].copy()

    sample = subset.sample(
        n=n,
        random_state=SEED
    )

    sample["sampling_source"] = "enriched"
    sample["target_candidate"] = "service"

    selected_parts.append(sample)

    CURRENT_USED.update(
        sample["id"].astype(str)
    )

# ------------------------------------------------------------
# 7. Ajouter 20 avis naturels
# 10 FR + 10 EN
# ------------------------------------------------------------

natural_parts = []

for language in ["fr", "en"]:

    subset = full_pool[
        (full_pool["language"] == language)
        & ~full_pool["id"].astype(str).isin(CURRENT_USED)
    ].copy()

    sample = subset.sample(
        n=10,
        random_state=SEED
    )

    sample["sampling_source"] = "natural"
    sample["target_candidate"] = "none"

    natural_parts.append(sample)

    CURRENT_USED.update(
        sample["id"].astype(str)
    )

# ------------------------------------------------------------
# 8. Assemblage final
# ------------------------------------------------------------

gold_v2_batch2 = pd.concat(
    selected_parts + natural_parts,
    ignore_index=True
)

gold_v2_batch2 = gold_v2_batch2.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

gold_v2_batch2 = gold_v2_batch2[
    [
        "id",
        "language",
        "text",
        "stars",
        "sampling_source",
        "target_candidate"
    ]
].copy()

# ------------------------------------------------------------
# 9. Contrôles
# ------------------------------------------------------------

print("\nShape :", gold_v2_batch2.shape)

print("\nLangues :")
print(
    gold_v2_batch2["language"]
    .value_counts()
)

print("\nSources :")
print(
    gold_v2_batch2["sampling_source"]
    .value_counts()
)

print("\nTargets :")
print(
    gold_v2_batch2["target_candidate"]
    .value_counts()
)

print(
    "\nIDs dupliqués :",
    gold_v2_batch2["id"]
    .duplicated()
    .sum()
)

print(
    "Textes dupliqués :",
    gold_v2_batch2["text"]
    .duplicated()
    .sum()
)

overlap = (
    set(gold_v2_batch2["id"].astype(str))
    & USED_ALL
)

print(
    "Chevauchement avec données déjà étudiées :",
    len(overlap)
)


assert len(gold_v2_batch2) == 300
assert gold_v2_batch2["id"].duplicated().sum() == 0
assert gold_v2_batch2["text"].duplicated().sum() == 0
assert len(overlap) == 0

# ------------------------------------------------------------
# 10. Export
# ------------------------------------------------------------

BATCH2_PATH = (
    GOLD_DIR / "gold_v2_batch2_300_unannotated.csv"
)

gold_v2_batch2.to_csv(
    BATCH2_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "\nBatch 2 sauvegardé :",
    BATCH2_PATH.relative_to(PROJECT_ROOT)
)

IDs déjà utilisés : 650

Shape : (300, 6)

Langues :
language
fr    150
en    150
Name: count, dtype: int64

Sources :
sampling_source
enriched    280
natural      20
Name: count, dtype: int64

Targets :
target_candidate
service     120
price       100
delivery     60
none         20
Name: count, dtype: int64

IDs dupliqués : 0
Textes dupliqués : 0
Chevauchement avec données déjà étudiées : 0

Batch 2 sauvegardé : data\gold\gold_v2_batch2_300_unannotated.csv


## 6. Fusion du Gold final et validation

Les deux batchs annotés sont fusionnés pour produire le **Gold V2 final de 700 avis**.

Avant le split, on vérifie :

- unicité des IDs ;
- doublons de texte ;
- absence de valeurs manquantes ;
- validité des quatre états ABSA.

> Dépendance : `iterative-stratification` doit être installée dans l'environnement du projet.
> L'installation n'est volontairement pas effectuée dans ce notebook.


In [25]:
# ============================================================
# 1. CHARGER LES DEUX BATCHS ANNOTÉS
# ============================================================

BATCH1_PATH = GOLD_DIR / "gold_v2_batch1_400_annotated_semantic_v2.csv"
BATCH2_PATH = GOLD_DIR / "gold_v2_batch2_300_annotated_semantic_v2.csv"

batch1 = pd.read_csv(BATCH1_PATH)
batch2 = pd.read_csv(BATCH2_PATH)

print("Batch 1 :", batch1.shape)
print("Batch 2 :", batch2.shape)

Batch 1 : (400, 10)
Batch 2 : (300, 10)


In [26]:
# ============================================================
# 2. FUSION GOLD V2
# ============================================================

gold_v2 = pd.concat(
    [batch1, batch2],
    ignore_index=True
)

print("Gold total :", gold_v2.shape)

print(
    "IDs dupliqués :",
    gold_v2["id"].duplicated().sum()
)

print(
    "Textes dupliqués :",
    gold_v2["text"].duplicated().sum()
)

print("\nLangues :")
print(gold_v2["language"].value_counts())
assert len(gold_v2) == 700
assert gold_v2["id"].duplicated().sum() == 0
assert gold_v2["text"].duplicated().sum() == 0


Gold total : (700, 10)
IDs dupliqués : 0
Textes dupliqués : 0

Langues :
language
fr    350
en    350
Name: count, dtype: int64


In [27]:
# ============================================================
# 3. VALIDATION DES LABELS
# ============================================================

ASPECTS = [
    "quality",
    "price",
    "delivery",
    "service",
]

VALID_LABELS = {
    "absent",
    "négatif",
    "neutre",
    "positif",
}

for aspect in ASPECTS:

    invalid = set(
        gold_v2[aspect].dropna().unique()
    ) - VALID_LABELS

    print(
        f"{aspect:10} | labels invalides :",
        invalid
    )

    assert not invalid, (
        f"Labels invalides détectés dans {aspect}: {invalid}"
    )

    assert gold_v2[aspect].isna().sum() == 0, (
        f"Valeurs manquantes dans {aspect}"
    )

print("\nTous les labels sont valides.")

quality    | labels invalides : set()
price      | labels invalides : set()
delivery   | labels invalides : set()
service    | labels invalides : set()

Tous les labels sont valides.


In [28]:
# ============================================================
# 4. MATRICE MULTILABEL POUR LE SPLIT
# ============================================================

stratification_parts = []

# Langue
language_dummies = pd.get_dummies(
    gold_v2["language"],
    prefix="language"
)

stratification_parts.append(
    language_dummies
)

# Aspect × label
for aspect in ASPECTS:

    aspect_dummies = pd.get_dummies(
        gold_v2[aspect],
        prefix=aspect
    )

    stratification_parts.append(
        aspect_dummies
    )

Y = pd.concat(
    stratification_parts,
    axis=1
).astype(int)

print("Matrice de stratification :", Y.shape)

display(Y.head())

Matrice de stratification : (700, 18)


,language_en,language_fr,quality_absent,quality_neutre,quality_négatif,quality_positif,price_absent,price_neutre,price_négatif,price_positif,delivery_absent,delivery_neutre,delivery_négatif,delivery_positif,service_absent,service_neutre,service_négatif,service_positif
0,0,1,0,0,1,0,0,0,1,0,1,0,0,0,1,0,0,0
1,0,1,0,0,0,1,1,0,0,0,0,0,0,1,1,0,0,0
2,0,1,0,0,1,0,1,0,0,0,1,0,0,0,1,0,0,0
3,0,1,0,0,0,1,0,0,0,1,1,0,0,0,1,0,0,0
4,0,1,0,0,0,1,0,0,0,1,1,0,0,0,1,0,0,0


## 7. Split officiel DEV / TEST

Le Gold est séparé avec une **stratification multilabel** afin de préserver simultanément :

- la langue ;
- les distributions `absent / négatif / neutre / positif` ;
- les quatre aspects.

Split retenu :

- **DEV : 500 avis** — utilisé pour les expériences et l'analyse d'erreurs ;
- **TEST : 200 avis** — réservé à l'évaluation finale.


In [29]:
# ============================================================
# 5. SPLIT GOLD DEV / TEST
# ============================================================

SEED_SPLIT = 42

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=200,
    random_state=SEED_SPLIT
)

dev_idx, test_idx = next(
    splitter.split(
        gold_v2,
        Y
    )
)

gold_dev = (
    gold_v2
    .iloc[dev_idx]
    .copy()
    .reset_index(drop=True)
)

gold_test = (
    gold_v2
    .iloc[test_idx]
    .copy()
    .reset_index(drop=True)
)

print("DEV  :", gold_dev.shape)
print("TEST :", gold_test.shape)

DEV  : (500, 10)
TEST : (200, 10)


In [30]:
# ============================================================
# 6. CONTRÔLES DEV / TEST
# ============================================================

dev_ids = set(
    gold_dev["id"].astype(str)
)

test_ids = set(
    gold_test["id"].astype(str)
)

print(
    "Chevauchement DEV / TEST :",
    len(dev_ids & test_ids)
)

len(dev_ids & test_ids) == 0

assert len(gold_dev) == 500
assert len(gold_test) == 200

assert len(gold_dev) + len(gold_test) == 700

print("\nSplit valide.")

Chevauchement DEV / TEST : 0

Split valide.


### Note méthodologique

Les statistiques DEV/TEST affichées ci-dessous servent uniquement à vérifier que le split initial
préserve correctement les distributions du Gold. Elles appartiennent à la **phase de construction du Gold**.

Après ce contrôle et l'export final, le TEST est considéré comme gelé et ne doit plus être utilisé
dans les notebooks expérimentaux suivants pour prendre une décision ML.


In [31]:
# ============================================================
# 7. STATISTIQUES DEV / TEST
# ============================================================

def print_split_stats(df, name):

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("\nLangues :")
    print(df["language"].value_counts())

    for aspect in ASPECTS:

        print(
            f"\n--- {aspect.upper()} ---"
        )

        counts = (
            df[aspect]
            .value_counts()
            .reindex(
                [
                    "absent",
                    "négatif",
                    "neutre",
                    "positif",
                ],
                fill_value=0
            )
        )

        print(counts)


print_split_stats(
    gold_dev,
    "GOLD DEV — 500"
)

print_split_stats(
    gold_test,
    "GOLD TEST — 200"
)


GOLD DEV — 500

Langues :
language
fr    250
en    250
Name: count, dtype: int64

--- QUALITY ---
quality
absent     104
négatif    199
neutre      59
positif    138
Name: count, dtype: int64

--- PRICE ---
price
absent     344
négatif     39
neutre      50
positif     67
Name: count, dtype: int64

--- DELIVERY ---
delivery
absent     322
négatif    101
neutre      32
positif     45
Name: count, dtype: int64

--- SERVICE ---
service
absent     393
négatif     47
neutre      13
positif     47
Name: count, dtype: int64

GOLD TEST — 200

Langues :
language
fr    100
en    100
Name: count, dtype: int64

--- QUALITY ---
quality
absent     41
négatif    81
neutre     23
positif    55
Name: count, dtype: int64

--- PRICE ---
price
absent     138
négatif     15
neutre      20
positif     27
Name: count, dtype: int64

--- DELIVERY ---
delivery
absent     128
négatif     41
neutre      13
positif     18
Name: count, dtype: int64

--- SERVICE ---
service
absent     157
négatif     19
neutre     

### Règle de gel du TEST

À partir de ce split, `gold_v2_test_200.csv` est **gelé**.

Il ne doit pas être utilisé pour choisir :

- le teacher LLM ;
- le prompt ;
- l'architecture ;
- les hyperparamètres ;
- les seuils ;
- une stratégie de nettoyage ou de pseudo-labellisation.

Toutes les décisions expérimentales doivent être prises avec le DEV.


## 8. Export des artefacts finaux

Trois artefacts sont produits :

- `gold_v2_700_with_provenance.csv` — version complète et traçable ;
- `gold_v2_dev_500.csv` — référence de développement ;
- `gold_v2_test_200.csv` — benchmark final gelé.

Les colonnes de provenance sont conservées dans la version complète mais retirées des
versions ML afin qu'elles ne puissent pas être utilisées comme features.


In [ ]:
# ============================================================
# 8. VERSION GOLD COMPLÈTE AVEC PROVENANCE
# ============================================================

GOLD_FULL_PROVENANCE_PATH = (
    GOLD_DIR / "gold_v2_700_with_provenance.csv"
)

gold_v2.to_csv(
    GOLD_FULL_PROVENANCE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Gold avec provenance :",
    GOLD_FULL_PROVENANCE_PATH.relative_to(PROJECT_ROOT)
)

Gold avec provenance : data\gold\gold_v2_700_with_provenance.csv


In [ ]:
# ============================================================
# 9. VERSION ML PROPRE
# ============================================================

ML_COLUMNS = [
    "id",
    "language",
    "text",
    "quality",
    "price",
    "delivery",
    "service",
]

gold_dev_ml = gold_dev[
    ML_COLUMNS
].copy()

gold_test_ml = gold_test[
    ML_COLUMNS
].copy()

DEV_PATH = (
    GOLD_DIR / "gold_v2_dev_500.csv"
)

TEST_PATH = (
    GOLD_DIR / "gold_v2_test_200.csv"
)
gold_dev_ml.to_csv(
    DEV_PATH,
    index=False,
    encoding="utf-8-sig"
)
gold_test_ml.to_csv(
    TEST_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("DEV  :", DEV_PATH.relative_to(PROJECT_ROOT))
print("TEST :", TEST_PATH.relative_to(PROJECT_ROOT))

DEV  : data\gold\gold_v2_dev_500.csv
TEST : data\gold\gold_v2_test_200.csv


In [35]:
# ============================================================
# 10. CHECKPOINT FINAL GOLD V2
# ============================================================

print("=== GOLD V2 FINALISÉ ===")

print(
    "Gold total :",
    len(gold_v2)
)

print(
    "Gold DEV   :",
    len(gold_dev_ml)
)

print(
    "Gold TEST  :",
    len(gold_test_ml)
)

print(
    "Overlap    :",
    len(
        set(gold_dev_ml["id"])
        & set(gold_test_ml["id"])
    )
)

print("\nLe TEST est maintenant GELÉ.")

=== GOLD V2 FINALISÉ ===
Gold total : 700
Gold DEV   : 500
Gold TEST  : 200
Overlap    : 0

Le TEST est maintenant GELÉ.


## Conclusion

La reconstruction du Gold Standard V2 est terminée.

Le projet dispose désormais de **700 avis bilingues annotés ABSA**, avec quatre aspects
indépendants et une distinction explicite entre aspect absent et sentiment neutre.

Le Gold est séparé en :

- **DEV 500** : développement expérimental ;
- **TEST 200** : évaluation finale gelée.

La prochaine phase du projet est la **pseudo-labellisation ABSA V2**.
Elle doit démarrer dans un notebook séparé afin de préserver une frontière claire entre
construction de la vérité terrain et production des données d'entraînement.
